In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

print("--- Initializing HC-WCI Machine Learning Module ---")

# 1. Load the Data
try:
    df = pd.read_csv('HCWCI_Master_Dataset.csv')
    print(f"Dataset loaded successfully. Total Cohorts: {len(df):,}\n")
except FileNotFoundError:
    print("Error: 'HCWCI_Master_Dataset.csv' not found. Please ensure it is in the same folder.")
    exit()

# 2. Preprocessing
# Ordinal encoding for Education Tier (E)
edu_map = {
    "1_Low (No/Primary)": 1,
    "2_Some High School": 2,
    "3_HS Graduate": 3,
    "4_Some College": 4,
    "5_College+ (Bachelor or Higher)": 5
}
df['Edu_Tier'] = df['Education_Group'].map(edu_map)

# Drop any rows with NaN values that might have snuck through
df = df.dropna(subset=['Years_in_US', 'Edu_Tier', 'Rent_Performance_Ratio', 'Avg_Income', 'Mortgage_Rate_Pct'])

# 3. Define the Features (X) and Target (y)
# Target Variable
y = df['Mortgage_Rate_Pct']

# Model A: The HC-WCI Framework (Alternative Data Only)
X_rf = df[['Years_in_US', 'Edu_Tier', 'Rent_Performance_Ratio']]

# Model B: The Baseline (Income Only)
X_base = df[['Avg_Income']]

# 4. Data Partitioning (80/20 Split with Seed 42)
# Split for Random Forest
X_train_rf, X_test_rf, y_train, y_test = train_test_split(X_rf, y, test_size=0.2, random_state=42)

# Split for Baseline (using the exact same index to ensure a fair fight)
X_train_base, X_test_base, _, _ = train_test_split(X_base, y, test_size=0.2, random_state=42)

print(f"Training set: {len(y_train):,} cohorts")
print(f"Testing set:  {len(y_test):,} cohorts\n")

# ==========================================
# 5. TRAIN MODEL A: Random Forest (HC-WCI)
# ==========================================
print("Training Random Forest (HC-WCI)...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_rf, y_train)

# Predict and Evaluate
rf_predictions = rf_model.predict(X_test_rf)
rf_r2 = r2_score(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))

# ==========================================
# 6. TRAIN MODEL B: Linear Regression (Baseline)
# ==========================================
print("Training Linear Regression (Income Baseline)...")
base_model = LinearRegression()
base_model.fit(X_train_base, y_train)

# Predict and Evaluate
base_predictions = base_model.predict(X_test_base)
base_r2 = r2_score(y_test, base_predictions)
base_rmse = np.sqrt(mean_squared_error(y_test, base_predictions))

# ==========================================
# 7. PRINT FINAL RESULTS FOR THE PAPER
# ==========================================
print("\n" + "="*50)
print("FINAL MODEL PERFORMANCE EVALUATION")
print("="*50)

print("\nBASELINE (Income Only):")
print(f"R-Squared (Accuracy): {base_r2:.4f}")
print(f"RMSE (Error Rate):    {base_rmse:.4f}")

print("\nHC-WCI RANDOM FOREST (Alternative Data):")
print(f"R-Squared (Accuracy): {rf_r2:.4f}")
print(f"RMSE (Error Rate):    {rf_rmse:.4f}")

print("\n" + "-"*50)
print("HC-WCI FEATURE IMPORTANCE (Gini Impurity)")
print("-"*50)
features = X_rf.columns
importances = rf_model.feature_importances_

for name, imp in zip(features, importances):
    # This formats the output so it looks clean for your paper
    print(f"{name:25}: {(imp * 100):.2f}% weight")
print("="*50)

--- Initializing HC-WCI Machine Learning Module ---
Dataset loaded successfully. Total Cohorts: 45,712

Training set: 36,569 cohorts
Testing set:  9,143 cohorts

Training Random Forest (HC-WCI)...
Training Linear Regression (Income Baseline)...

FINAL MODEL PERFORMANCE EVALUATION

BASELINE (Income Only):
R-Squared (Accuracy): 0.0285
RMSE (Error Rate):    33.6616

HC-WCI RANDOM FOREST (Alternative Data):
R-Squared (Accuracy): 0.2906
RMSE (Error Rate):    28.7651

--------------------------------------------------
HC-WCI FEATURE IMPORTANCE (Gini Impurity)
--------------------------------------------------
Years_in_US              : 13.34% weight
Edu_Tier                 : 6.49% weight
Rent_Performance_Ratio   : 80.17% weight
